# 02. Однопоточный CrossDiT K5: visual state и текстовый context

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Разберём SA→text CA→FFN, воспроизведём оригинальный K5 block и сравним его с соответствующим K6 single-stream block при одинаковых весах.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** Предпосылки MIT: Lab 3 Q3.3–Q3.5 и class-conditioning Parts 1–2. Text cross-attention / CrossDiT K5 — промышленное расширение; не выдаём его за отдельное MIT задание.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='02_single_stream_crossdit_k5';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Что означает «один поток»

Здесь один **генерируемый** latent stream V; text tokens T — внешний conditioning context, который может дополнительно обрабатываться text transformer/LTF. Это не утверждение, что в модели всего один tensor, и не то же самое, что joint self-attention по concatenated visual+text tokens.

$$
V_1=V+g_{sa}\odot\mathrm{SA}(\mathrm{M}_{sa}(V;c)),
$$

$$
V_2=V_1+g_{txt}\odot\mathrm{CA}(Q=\mathrm{M}_{txt}(V_1;c),K=T,V=T),
$$

$$
V_3=V_2+g_{ff}\odot\mathrm{FFN}(\mathrm{M}_{ff}(V_2;c)).
$$

Text CA: V shape B,N_v,D; T shape B,N_t,D после text projection. Q имеет N_v positions, K/V — N_t. Output сохраняет N_v. Нет требования N_v=N_t. В K6 multimodal CA дополнительно разрешены разные feature widths.

История: [DiT](https://arxiv.org/abs/2212.09748) — исходный diffusion-transformer framework; [PixArt-α](https://arxiv.org/abs/2310.00426) — текстовый cross-attention precedent. Слово CrossDiT не обозначает одного универсального блока всех авторов; здесь определяем его по K5/K6 source и не заявляем абсолютное первое употребление термина.


In [ ]:
k5=load_k5_block();old=k5.TransformerDecoderBlock(48,16,96,8,attention_engine='sdpa',text_token_padding=True)
for m in old.modules():
    if isinstance(m,k5.Modulation):nn.init.normal_(m.out_layer.weight,std=.08);nn.init.normal_(m.out_layer.bias,std=.08)
new=native().TransformerDecoderBlock(48,16,96,8,engine='sdpa',text_token_padding=True)
# Source modules have different FFN key names; map ONLY the equivalent FFN Linears.
state={}
for key,value in old.state_dict().items():
    key=key.replace('feed_forward.in_layer.','feed_forward.net.0.proj.').replace('feed_forward.out_layer.','feed_forward.net.2.')
    state[key]=value
new.load_state_dict(state,strict=True)
rv=native('core.components.rope').RoPE1D(8)(torch.arange(8))
v=torch.randn(1,8,48);text=torch.randn(1,5,48);c=torch.randn(1,16)
mask=torch.tensor([[[[True,True,True,True,False]]]])
with torch.no_grad():a=old(v,text,c,rv,None,mask);b=new(v,text,c,rv,mask);manual=manual_single(new,v,text,c,rv,mask)
assert torch.allclose(a,b,atol=2e-6) and torch.allclose(b,manual,atol=2e-6)
metrics['k5_k6_dense_block_parity_error']=float((a-b).abs().max())
metrics['k6_manual_sequence_error']=float((b-manual).abs().max())
print(metrics)


## 2. Что изменено в execution adapter

Исполняются native class bodies K5; CUDA compile/autocast decorators сняты через AST. Affine/residual/RoPE helpers заменены локальными FP32 helpers K6 с теми же математическими операциями. Dense attention backend реализует ожидаемый helper return protocol. Sparse/NABLA, BF16 rounding, distributed inference и whole-model K5↔K6 checkpoint parity не проверялись.

В одинаковой точности, с одинаковыми весами, порядок SA→CA→FFN даёт тот же small-block result. Это более узкое утверждение, чем «K5 и K6 полностью одинаковы».


In [ ]:
show_source('dit.py','TransformerDecoderBlock',family='k5')
show_source('core/components/dit.py','TransformerDecoderBlock')
changed=text.clone();changed[:,-1]+=1000
with torch.no_grad():masked=new(v,changed,c,rv,mask);unmasked=new(v,changed,c,rv,None)
assert torch.equal(masked,b)
metrics['masked_token_perturbation_error']=float((masked-b).abs().max())
metrics['unmasked_token_perturbation_error']=float((unmasked-b).abs().max())
assert metrics['unmasked_token_perturbation_error']>1e-5


## 3. Cross-attention и условная зависимость

При фиксированных weights/normalization текст влияет через K/V. При закрытом text gate эта **ветвь** не меняет V. Однако whole model может получать тот же prompt ещё через pooled embedding в modulation — поэтому закрыть CA недостаточно для полного удаления text conditioning.

Проверим два разных утверждения: влияние text tokens на block при фиксированном c и влияние pooled c при фиксированных tokens. Это разные causal interventions, не оценка prompt adherence модели.


In [ ]:
v0=v.clone();text2=text+.3*torch.randn_like(text);c2=c+.2*torch.randn_like(c)
with torch.no_grad():
    base=new(v0,text,c,rv,mask)
    token_change=new(v0,text2,c,rv,mask)
    pooled_change=new(v0,text,c2,rv,mask)
metrics['token_path_change']=float((token_change-base).square().mean())
metrics['modulation_path_change']=float((pooled_change-base).square().mean())
assert metrics['token_path_change']>0 and metrics['modulation_path_change']>0
fig,ax=plt.subplots(figsize=(6,3));ax.bar(['Token context','Time/pooled vector'],[metrics['token_path_change'],metrics['modulation_path_change']]);ax.set(ylabel='Output MSE to baseline',title='Two distinct conditioning routes in random block')
fig.tight_layout();fig.savefig(figure_path(LAB,'conditioning_routes'));plt.show()


## Измерение → вывод

Original dense K5 block и K6 single-stream counterpart совпали при явном parameter mapping/FP32 adapter. Masked key perturbation не повлияла на output. Text CA и pooled modulation являются разными маршрутами условия.


## Самостоятельные опыты

1. Сопоставьте MIT class embedding с pooled branch и объясните, чего ему недостаёт для token CA.
2. Закройте только CA gate и повторите два interventions.
3. Выпишите Q/K/V и complexity для N_v=100,N_t=20,D=48,h=6.
4. Почему измеренная small-block parity не доказывает whole-checkpoint portability?

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
